# Unified SigExt Training Pipeline
Trains all 4 SigExt configurations and uploads to HuggingFace.

## 1. Setup

In [1]:
%%capture
!pip install transformers datasets accelerate sentence-transformers \
    spacy huggingface_hub "numpy<2.0" "scipy>=1.10"
!python -m spacy download it_core_news_sm

In [2]:
import os
import json
import gc
import torch
import torch.nn as nn
import spacy
import numpy as np
from tqdm.auto import tqdm
from torch.utils.data import Dataset
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    Trainer, 
    TrainingArguments
)
from huggingface_hub import login, HfApi

## 2. Configuration

In [3]:
# Auth
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except:
    HF_TOKEN = os.getenv("HF_TOKEN") or "YOUR_HF_TOKEN_HERE"

login(token=HF_TOKEN)

HF_USERNAME = "LookUpMark"

# All training configurations
TRAIN_CONFIGS = {
    "10k-060t": {
        "num_samples": 10000,
        "threshold": 0.60,
        "repo_name": "sigext-wits-it-10k-060t"
    },
    "25k-060t": {
        "num_samples": 25000,
        "threshold": 0.60,
        "repo_name": "sigext-wits-it-25k-060t"
    },
    "25k-065t": {
        "num_samples": 25000,
        "threshold": 0.65,
        "repo_name": "sigext-wits-it-25k-065t"
    },
    "25k-070t": {
        "num_samples": 25000,
        "threshold": 0.70,
        "repo_name": "sigext-wits-it-25k-070t"
    }
}

# Global settings
GLOBAL_CONFIG = {
    "sbert_model": "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    "longformer_model": "markussagen/xlm-roberta-longformer-base-4096",
    "max_len": 2048,
    "min_source_len": 500,
    "max_source_len": 10000,
    "min_summary_len": 50,
    "min_sent_len": 20,
    "batch_size": 32,
    "output_dir": "./models"
}

os.makedirs(GLOBAL_CONFIG["output_dir"], exist_ok=True)
print(f"Training {len(TRAIN_CONFIGS)} configurations")

Training 4 configurations


## 3. Dataset Classes

In [4]:
class SigExtDataset(Dataset):
    """
    Dataset with CORRECT sentence-to-token label alignment.
    Each token inherits the label of its parent sentence.
    """
    
    def __init__(self, data_path: str, tokenizer, max_len: int):
        self.data = [json.loads(line) for line in open(data_path)]
        self.tokenizer = tokenizer
        self.max_len = max_len
        print(f"Loaded {len(self.data)} training examples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        sentences = item['sentences']
        sent_labels = item['labels']
        
        # Tokenize each sentence separately to track boundaries
        all_input_ids = []
        all_labels = []
        
        for sent, label in zip(sentences, sent_labels):
            # Tokenize sentence (without special tokens)
            tokens = self.tokenizer.encode(sent, add_special_tokens=False)
            all_input_ids.extend(tokens)
            # Assign same label to ALL tokens of this sentence
            all_labels.extend([label] * len(tokens))
        
        # Add special tokens
        # [CLS] at start, [SEP] at end
        cls_id = self.tokenizer.cls_token_id or self.tokenizer.bos_token_id
        sep_id = self.tokenizer.sep_token_id or self.tokenizer.eos_token_id
        
        all_input_ids = [cls_id] + all_input_ids + [sep_id]
        all_labels = [-100] + all_labels + [-100]  # Ignore special tokens
        
        # Truncate
        if len(all_input_ids) > self.max_len:
            all_input_ids = all_input_ids[:self.max_len]
            all_labels = all_labels[:self.max_len]
        
        # Pad
        pad_len = self.max_len - len(all_input_ids)
        attention_mask = [1] * len(all_input_ids) + [0] * pad_len
        all_input_ids = all_input_ids + [self.tokenizer.pad_token_id] * pad_len
        all_labels = all_labels + [-100] * pad_len  # Ignore padding
        
        return {
            "input_ids": torch.tensor(all_input_ids),
            "attention_mask": torch.tensor(attention_mask),
            "labels": torch.tensor(all_labels)
        }


class WeightedTrainer(Trainer):
    """Trainer with weighted loss for class imbalance."""
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        device = inputs["input_ids"].device

        # Weight: non-salient=1.0, salient=10.0
        class_weights = torch.tensor([1.0, 10.0]).to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100)
        
        loss = criterion(
            outputs.get("logits").view(-1, 2),
            labels.view(-1)
        )
        
        return (loss, outputs) if return_outputs else loss

## 4. Helper Functions

In [5]:
def prepare_training_data(config_name, num_samples, threshold):
    """Generate semantically labeled training data."""
    nlp = spacy.load("it_core_news_sm")
    
    print(f"Loading SBERT...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    sbert = SentenceTransformer(GLOBAL_CONFIG["sbert_model"], device=device)
    
    print(f"Loading WITS dataset...")
    dataset = load_dataset("silvia-casola/WITS", split="train", streaming=True)
    
    output_file = f"train_data_{config_name}.jsonl"
    count = 0
    
    with open(output_file, "w") as f_out:
        pbar = tqdm(total=num_samples, desc=f"Labeling {config_name}")
        
        for entry in dataset:
            source = entry['source']
            summary = entry['summary']
            
            # Quality filters
            if (len(source) < GLOBAL_CONFIG["min_source_len"] or 
                len(summary) < GLOBAL_CONFIG["min_summary_len"] or 
                len(source) > GLOBAL_CONFIG["max_source_len"]):
                continue

            # Segment into sentences
            doc_sents = [s.text for s in nlp(source).sents 
                        if len(s.text) > GLOBAL_CONFIG["min_sent_len"]]
            sum_sents = [s.text for s in nlp(summary).sents 
                        if len(s.text) > GLOBAL_CONFIG["min_sent_len"]]

            if not doc_sents or not sum_sents:
                continue

            # Compute SBERT embeddings
            doc_emb = sbert.encode(doc_sents, convert_to_tensor=True, 
                                   show_progress_bar=False, 
                                   batch_size=GLOBAL_CONFIG["batch_size"])
            sum_emb = sbert.encode(sum_sents, convert_to_tensor=True, 
                                   show_progress_bar=False, 
                                   batch_size=GLOBAL_CONFIG["batch_size"])
            
            # Compute similarity and generate labels
            scores = util.cos_sim(doc_emb, sum_emb)
            labels = [
                1 if scores[i].max().item() > threshold else 0
                for i in range(len(doc_sents))
            ]
            
            # Keep only if at least one salient sentence
            if 1 in labels:
                f_out.write(json.dumps({
                    "sentences": doc_sents, 
                    "labels": labels
                }) + "\n")
                count += 1
                pbar.update(1)
            
            if count >= num_samples:
                break
        
        pbar.close()

    print(f"Saved {count} samples to {output_file}")
    
    del sbert
    torch.cuda.empty_cache()
    gc.collect()
    
    return output_file


def train_model(data_file, config_name):
    """Train SigExt model on prepared data."""
    print(f"Loading Longformer...")
    
    # Use Longformer tokenizer
    tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
    model = AutoModelForTokenClassification.from_pretrained(
        GLOBAL_CONFIG["longformer_model"], 
        num_labels=2
    )

    args = TrainingArguments(
        output_dir=f"./checkpoints_{config_name}",
        num_train_epochs=2,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-5,
        fp16=True,
        save_strategy="epoch",
        logging_steps=100,
        report_to="none"
    )
    
    dataset = SigExtDataset(data_file, tokenizer, GLOBAL_CONFIG["max_len"])
    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=dataset
    )
    
    print(f"Training {config_name}...")
    trainer.train()
    
    # Save
    output_path = os.path.join(GLOBAL_CONFIG["output_dir"], config_name)
    model.save_pretrained(output_path)
    tokenizer.save_pretrained(output_path)
    print(f"Saved to {output_path}")
    
    del model, trainer
    torch.cuda.empty_cache()
    gc.collect()
    
    return output_path


def upload_to_hub(model_path, repo_name):
    """Upload model to HuggingFace Hub (overwrites existing)."""
    repo_id = f"{HF_USERNAME}/{repo_name}"
    api = HfApi()
    
    try:
        api.create_repo(repo_id=repo_id, exist_ok=True)
    except Exception as e:
        print(f"Repo error: {e}")

    api.upload_folder(
        folder_path=model_path,
        repo_id=repo_id,
        repo_type="model",
        commit_message=f"Retrained with fixed label alignment"
    )
    print(f"Uploaded: https://huggingface.co/{repo_id}")

## 5. Training Loop

In [6]:
print("="*60)
print("UNIFIED SIGEXT TRAINING PIPELINE")
print("="*60)

for config_name, config in TRAIN_CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"CONFIG: {config_name}")
    print(f"  Samples: {config['num_samples']}")
    print(f"  Threshold: {config['threshold']}")
    print(f"{'='*60}")
    
    # Step 1: Generate labeled data
    data_file = prepare_training_data(
        config_name,
        config['num_samples'],
        config['threshold']
    )
    
    # Step 2: Train model
    model_path = train_model(data_file, config_name)
    
    # Step 3: Upload to HuggingFace
    upload_to_hub(model_path, config['repo_name'])
    
    # Cleanup data file
    os.remove(data_file)
    print(f"Completed: {config_name}")

print("\n" + "="*60)
print("ALL TRAINING COMPLETE!")
print("="*60)

UNIFIED SIGEXT TRAINING PIPELINE

CONFIG: 10k-060t
  Samples: 10000
  Threshold: 0.6
Loading SBERT...
Loading WITS dataset...


Labeling 10k-060t:   0%|          | 0/10000 [00:00<?, ?it/s]

Saved 10000 samples to train_data_10k-060t.jsonl
Loading Longformer...


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

...

Step,Training Loss
100,0.669800
200,0.647400
300,0.625700
400,0.620900
500,0.611900
600,0.615300
700,0.619100
800,0.599900
900,0.594400
1000,0.602100


Saved to ./models/25k-070t


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploaded: https://huggingface.co/LookUpMark/sigext-wits-it-25k-070t
Completed: 25k-070t

ALL TRAINING COMPLETE!
